# Time Series Forecast Model B Workflow

Self-contained duration forecasting workflow. This notebook defines its data-building, preprocessing, classical ML, and LSTM steps inline and does not import helper code from other project files.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from pandas.tseries.holiday import USFederalHolidayCalendar


@dataclass
class TimeSeriesForecastConfig:
    data_path: str = "data/ChargePointData2011-2020-ManullyPreCleaned3.csv"
    model_dir: str = "model_B"
    test_size: float = 0.2
    time_steps: int = 12


def to_seconds_vectorized(series):
    """Convert hh:mm:ss strings to total seconds."""
    temp = series.astype(str).str.split(':', expand=True).astype(float)
    return (temp[0] * 3600) + (temp[1] * 60) + temp[2]


def get_palo_alto_weather(start_date, end_date):
    """Fetch historical hourly weather for Palo Alto. Returns an empty frame if the API is unavailable."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 37.4419,
        "longitude": -122.1430,
        "start_date": start_date.strftime('%Y-%m-%d'),
        "end_date": end_date.strftime('%Y-%m-%d'),
        "hourly": ["temperature_2m", "precipitation", "cloudcover"],
        "timezone": "America/Los_Angeles",
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        res_data = response.json()
        return pd.DataFrame({
            'weather_time': pd.to_datetime(res_data['hourly']['time']),
            'temp': res_data['hourly']['temperature_2m'],
            'precip': res_data['hourly']['precipitation'],
            'clouds': res_data['hourly']['cloudcover'],
        })
    except Exception as exc:
        print(f"Weather API unavailable ({exc}). Continuing without weather columns.")
        return pd.DataFrame()


def engineer_time_features(df):
    """Extract calendar and cyclical time features from Start Date."""
    df = df.copy()
    df['Start Date'] = pd.to_datetime(df['Start Date'])
    df['Minute'] = df['Start Date'].dt.minute
    df['Hour'] = df['Start Date'].dt.hour
    df['DayOfWeek'] = df['Start Date'].dt.dayofweek
    df['Month'] = df['Start Date'].dt.month
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['Start Date'].min(), end=df['Start Date'].max())
    df['IsHoliday'] = df['Start Date'].dt.normalize().isin(holidays).astype(int)

    df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    return df


def build_final_dataset(data, available=False):
    """Build Model B's duration forecasting dataset from raw charging sessions."""
    df = data.copy()
    df['Start Date'] = pd.to_datetime(df['Start Date'])
    df['End Date'] = pd.to_datetime(df['End Date'])

    full_df = df[['MAC Address', 'Start Date', 'End Date', 'Plug Type', 'Address 1']].copy()
    full_df['Total_Duration_Secs'] = to_seconds_vectorized(df['Total Duration (hh:mm:ss)'])
    full_df = full_df.sort_values('Start Date').reset_index(drop=True)

    print("Fetching Palo Alto weather data...")
    weather = get_palo_alto_weather(full_df['Start Date'].min(), full_df['Start Date'].max())
    if not weather.empty:
        full_df['merge_hour'] = full_df['Start Date'].dt.floor('h')
        full_df = full_df.merge(weather, left_on='merge_hour', right_on='weather_time', how='left')
        full_df.drop(columns=['merge_hour', 'weather_time'], inplace=True)
    else:
        full_df['temp'] = np.nan
        full_df['precip'] = np.nan
        full_df['clouds'] = np.nan

    full_df = engineer_time_features(full_df)
    full_df['Plug Type Encoded'] = full_df['Plug Type'].map({'J1772': 0, 'NEMA 5-20R': 1}).fillna(0).astype(int)

    cols_to_drop = ['Start Date', 'End Date', 'Plug Type', 'Address 1']
    final_df = full_df.drop(columns=cols_to_drop)
    final_df = final_df.fillna(final_df.mean(numeric_only=True))

    print(f"Final dataset shape: {final_df.shape}")
    print(f"Duration target summary:\n{final_df['Total_Duration_Secs'].describe()}")
    return final_df


class TimeSeriesForecastPipeline:
    def __init__(self, config):
        self.config = config
        self.raw_data = None
        self.final_df = None

    def load_data(self):
        self.raw_data = pd.read_csv(self.config.data_path, low_memory=False)
        return self.raw_data

    def build_dataset(self, available=False):
        if self.raw_data is None:
            self.load_data()
        self.final_df = build_final_dataset(self.raw_data, available=available)
        return self.final_df


config = TimeSeriesForecastConfig(
    data_path="data/ChargePointData2011-2020-ManullyPreCleaned3.csv",
    model_dir="model_B",
    test_size=0.2,
    time_steps=12,
)

Path(config.model_dir).mkdir(exist_ok=True)
pipeline = TimeSeriesForecastPipeline(config)

## 1. Build the Final Dataset

In [ ]:
pipeline.load_data()
final_df = pipeline.build_dataset(available=False)

final_df.head()

In [ ]:
final_df.info()

## 2. Split the Label

In [ ]:
split_idx = int(len(final_df) * 0.8)
train_df = final_df.iloc[:split_idx]
test_df = final_df.iloc[split_idx:]
train_df

## 3. Pipeline the Data

In [ ]:
train_df_label_duration = train_df['Total_Duration_Secs'].copy()
train_df_data = train_df.drop(['Total_Duration_Secs'], axis = 1)
test_df_label_duration = test_df['Total_Duration_Secs'].copy()
test_df_data = test_df.drop(['Total_Duration_Secs'], axis = 1)
print("Train data shape:", train_df_data.shape)
print("Duration label shape:", train_df_label_duration.shape)
print("Test data shape:", test_df_data.shape)
print("Test duration label shape:", test_df_label_duration.shape)

In [ ]:
train_df_data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_attribs = list(train_df_data.select_dtypes(include="number").columns)
cat_attribs = ['MAC Address']

try:
    mac_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    mac_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('mac_onehot', mac_encoder, cat_attribs),
        ('std_scaler', StandardScaler(), num_attribs),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
])

train_df_prepared = full_pipeline.fit_transform(train_df_data, train_df_label_duration)
feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
train_df_prepared = pd.DataFrame(train_df_prepared, columns=feature_names, index=train_df_data.index)
train_df_prepared.head()

## 4. Train ML Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

lr = LinearRegression()
lr_scores = cross_val_score(lr, train_df_prepared, train_df_label_duration, scoring="neg_mean_absolute_error", cv=5)
print("Scores:", lr_scores)

In [ ]:
import xgboost as xgb

xgb_rg = xgb.XGBRegressor()
xgb_rg.fit(train_df_prepared, train_df_label_duration)
xgb_rg_scores = cross_val_score(xgb_rg, train_df_prepared, train_df_label_duration, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1)
print("Scores:", xgb_rg_scores)
# Check recall and precision as well, SAVE THE MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

param_grid_lr = {
    "fit_intercept": [True, False],
    "positive": [True, False] # Forces coefficients to be positive if True
}

param_grid_dt = {
    "criterion": ["squared_error", "friedman_mse"],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10]
}

param_grid_rf = {
    "n_estimators": [100, 200],
    "criterion": ["squared_error"],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}

models = [
    ("LinearRegression", LinearRegression(), param_grid_lr),
    ("DecisionTree", DecisionTreeRegressor(random_state=42), param_grid_dt),
    ("RandomForest", RandomForestRegressor(random_state=42), param_grid_rf),
    ("XGBRegressor", xgb.XGBRegressor(eval_metric="rmse", random_state=42), param_grid_xgb),
]

best_results = {}
saved = False
if not saved:
    for name, estimator, param_grid in models:
        grid = GridSearchCV(estimator, param_grid, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1, verbose=1)
        grid.fit(train_df_prepared, train_df_label_duration)
        best_results[name] = {
            "best_score": grid.best_score_,
            "best_params": grid.best_params_,
            "best_estimator": grid.best_estimator_
        }
        print(f"{name}: best_score={grid.best_score_:.4f}")
        print("best_params=", grid.best_params_)
        print()

    best_results
# Random forest took the longest time


In [ ]:
import joblib
from pathlib import Path

model_dir = Path(config.model_dir)
model_dir.mkdir(exist_ok=True)

if 'best_results' in globals() and best_results:
    joblib.dump(best_results['XGBRegressor']['best_estimator'], model_dir / 'best_xgb_model.pkl')
    joblib.dump(best_results['XGBRegressor']['best_params'], model_dir / 'best_xgb_params.pkl')

    joblib.dump(best_results['RandomForest']['best_estimator'], model_dir / 'best_rf_model.pkl')
    joblib.dump(best_results['RandomForest']['best_params'], model_dir / 'best_rf_params.pkl')

    joblib.dump(best_results['DecisionTree']['best_estimator'], model_dir / 'best_dt_model.pkl')
    joblib.dump(best_results['DecisionTree']['best_params'], model_dir / 'best_dt_params.pkl')

    joblib.dump(best_results['LinearRegression']['best_estimator'], model_dir / 'best_lr_model.pkl')
    joblib.dump(best_results['LinearRegression']['best_params'], model_dir / 'best_lr_params.pkl')

best_xgb_model = joblib.load(model_dir / 'best_xgb_model.pkl')
best_xgb_params = joblib.load(model_dir / 'best_xgb_params.pkl')

best_rf_model = joblib.load(model_dir / 'best_rf_model.pkl')
best_rf_params = joblib.load(model_dir / 'best_rf_params.pkl')

best_dt_model = joblib.load(model_dir / 'best_dt_model.pkl')
best_dt_params = joblib.load(model_dir / 'best_dt_params.pkl')

best_lr_model = joblib.load(model_dir / 'best_lr_model.pkl')
best_lr_params = joblib.load(model_dir / 'best_lr_params.pkl')

In [ ]:
# Assemble
from sklearn.ensemble import VotingRegressor

xgb_best = best_xgb_model
lr_best = best_lr_model
rf_best = best_rf_model
dc_best= best_dt_model

voting_regressor = VotingRegressor(
    estimators=[('lr', lr_best), ('rf', rf_best), ('dc', dc_best), ('xgb', xgb_best)],
    n_jobs=-1
)
voting_regressor.fit(train_df_prepared, train_df_label_duration)

# Test Set

In [ ]:
final_test_df_data = full_pipeline.transform(test_df_data)
final_test_df_data = pd.DataFrame(final_test_df_data, columns=feature_names, index=test_df_data.index)

In [ ]:
final_test_df_data

In [ ]:
from sklearn.metrics import mean_absolute_error

for reg in (lr_best, rf_best, dc_best, xgb_best, voting_regressor):
    y_pred = reg.predict(final_test_df_data)
    print(reg.__class__.__name__, mean_absolute_error(test_df_label_duration, y_pred))

## 5. Train LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, LSTM
from tensorflow.keras.losses import Huber
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)


def create_grouped_sequences(feature_df, target_series, group_series, time_steps=12):
    """Create fixed-length histories per charger so LSTM windows do not mix MAC addresses."""
    X, y = [], []
    temp = feature_df.copy()
    temp['_target'] = target_series.loc[feature_df.index].values
    temp['_group'] = group_series.loc[feature_df.index].values

    for _, group in temp.groupby('_group', sort=False):
        group = group.drop(columns=['_group'])
        if len(group) <= time_steps:
            continue

        feature_values = group.drop(columns=['_target']).to_numpy(dtype='float32')
        target_values = group['_target'].to_numpy(dtype='float32')
        for i in range(time_steps, len(group)):
            X.append(feature_values[i - time_steps:i])
            y.append(target_values[i])

    return np.asarray(X, dtype='float32'), np.asarray(y, dtype='float32')


target_scaler = StandardScaler()
y_train_scaled = pd.Series(
    target_scaler.fit_transform(train_df_label_duration.to_frame()).ravel(),
    index=train_df_label_duration.index,
    name='Total_Duration_Secs_scaled',
)
y_test_scaled = pd.Series(
    target_scaler.transform(test_df_label_duration.to_frame()).ravel(),
    index=test_df_label_duration.index,
    name='Total_Duration_Secs_scaled',
)

X_train_seq, y_train_seq = create_grouped_sequences(
    train_df_prepared,
    y_train_scaled,
    train_df_data['MAC Address'],
    time_steps=config.time_steps,
)
X_test_seq, y_test_seq = create_grouped_sequences(
    final_test_df_data,
    y_test_scaled,
    test_df_data['MAC Address'],
    time_steps=config.time_steps,
)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("X_test_seq:", X_test_seq.shape)
print("y_test_seq:", y_test_seq.shape)

In [ ]:
def build_lstm_regressor(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.30),
        LSTM(64),
        BatchNormalization(),
        Dropout(0.25),
        Dense(64, activation='relu'),
        Dropout(0.15),
        Dense(1, activation='linear'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=Huber(delta=1.0),
        metrics=['mae'],
    )
    return model


lstm_model = build_lstm_regressor((X_train_seq.shape[1], X_train_seq.shape[2]))
lstm_model.summary()

In [ ]:
lstm_model_path = Path(config.model_dir) / 'lstm_duration.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    ModelCheckpoint(lstm_model_path, monitor='val_loss', save_best_only=True),
]

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
y_pred_scaled = lstm_model.predict(X_test_seq).ravel()
y_pred_duration = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_true_duration = target_scaler.inverse_transform(y_test_seq.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_true_duration, y_pred_duration)
rmse = mean_squared_error(y_true_duration, y_pred_duration, squared=False)
r2 = r2_score(y_true_duration, y_pred_duration)

print(f"LSTM MAE: {mae:,.2f} seconds")
print(f"LSTM RMSE: {rmse:,.2f} seconds")
print(f"LSTM R2: {r2:.4f}")
print(f"Saved best LSTM model to: {lstm_model_path}")